In [23]:
import json
import random
import re
import ast
import random
from collections import defaultdict
from pathlib import Path
from transformers import AutoTokenizer

ABS_REL_DIR = Path("/home/lucas/Desktop/UCSD/Research/sequential-decision-processors/verl_dead_agent/agent_system/environments")
tok_path = ABS_REL_DIR / "tokenizers" / "qwen3"
tok = AutoTokenizer.from_pretrained(tok_path, use_fast=True, local_files_only=True)

In [24]:
def load_jsonl(path: Path):
    data = []
    jsonl_files = sorted(path.glob("*.jsonl"), key=lambda x: int(x.stem) if x.stem.isdigit() else float('inf'))    
    for jsonl_file in jsonl_files:
        with jsonl_file.open("r", encoding="utf-8") as f:
            for line in f:
                if not line:
                    continue
                sample = json.loads(line)
                if sample.get('score', 0) >= 1:
                    data.append(sample)
    
    print(f"Loaded {len(data)} samples from {len(jsonl_files)} files.")
    return data

# load our data in
data = load_jsonl(Path("rejection_sampling/alfworld_7"))

Loaded 287 samples from 2 files.


In [25]:
# # Optionally can print out a sample
# for k,v in data[0].items():
#     print(f"{k}\n{v}\n\n")

In [28]:
# Now converting data to my cleaned sft format
def process_and_write(data, filename, sys_prompt_name):
    cleaned_path = Path("cleaned_sft")
    cleaned_path.mkdir(exist_ok=True)
    out_file = cleaned_path / f"{filename}.jsonl"
    with out_file.open("w", encoding="utf-8") as f:
        for d in data:
            inp = d.get("input", "").strip()
            # Remove the starting 'user\n' and ending '\nassistant'
            inp = inp.removeprefix("user\n").removesuffix("\nassistant").strip()
            out = d.get("output", "").strip()

            chat = [
                ["system", sys_prompt_name],
                ["user", inp],
                ["assistant", out]
            ]

            json.dump({"chat": chat}, f, ensure_ascii=False)
            f.write("\n")

In [29]:
process_and_write(data, "rft_alfworld", sys_prompt_name="tw_general.txt")